# nano Beatrix — the output layer's dial: the aleph read as the confidence dial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_head_dial.ipynb)

**Why this runs.** With the output layer's readout on AdamW (the v1.1 recipe) the head's aleph read stays at its birth: switching it off costs +0.0003 to +0.0006 bits a byte and its weights stay noise-sized. A read born at zero beside a complete linear readout, on the same input and the same target, earns nothing the readout cannot write from the state. The one function a linear readout cannot write after the final LayerNorm is scale: the norm gives every byte one size, so the per-byte confidence rides on the last block's big direction instead, at the cost of the content's room. The dial gives that function to the aleph read: the readout's logits times `exp(ln4 · tanh(w_t · s(h) − ā))`, `w_t` born zero (the dial at exactly 1, the plain craft at birth), `ā` the running mean over the training forwards, the scale within [1/4, 4]; the carrying read stays frozen at zero. Each arm trains at the v1.1 recipe (Muon on the trunk, the readout on AdamW at decay 0.1) and reads against the readout-on-AdamW run of its own body and start:

| arm | body | start | head | its reference run |
|---|---|---|---|---|
| `c-dial-s3` | Castor | seed 3 | dial | `c-head-adamw0.1` |
| `c-dial-s4` | Castor | seed 4 | dial | `c-head-adamw0.1-s4` |
| `c-dialfrz-s3` | Castor | seed 3 | dial over a frozen random address (the control) | `c-head-adamw0.1` |
| `c-dialraw-s3` | Castor | seed 3 | dial without its running-mean centering | `c-head-adamw0.1` |
| `c-dialplus-s3` | Castor | seed 3 | dial with the carrying read kept beside it | `c-head-adamw0.1` |
| `c-pergain-s3` | Castor | seed 3 | a per-byte gain `G s(h)` (G under the readout's AdamW) | `c-head-adamw0.1` |
| `pdc-dial-s1` | Pollux's layout, dual hubs + Canon | seed 1 | dial | `pdc-head-adamw0.1` |

Everything else is the 2s recipe (Muon 0.02 on the other matrices, Adam 0.0003 without decay on the rest, 200 warmup steps then flat, clip 1.0, 262,144 bytes a step), the same data in the same order as the earlier screens, 4,960 steps. Each arm starts from its reference run's start, bit for bit, with the dial's tensors at zero; the reference runs are read from the repo.

**How an arm passes** (written before any arm runs; each arm against its reference run at 4,960 steps): (i) function: the head's toggle (`lastwrite/head_toggle_bits`, the dial held at 1) at least 0.010 bits a byte (the reference +0.0003); (ii) no cost: bits a byte within 0.0029 of the reference, or better; (iii) room: the share after the final norm below the reference's; (iv) a dial, not a gate: its log scale correlates positively with confidence (`head_dial_corr_entropy` above 0) and fewer than half the positions sit at the bound (`head_dial_sat` under 0.5). Both Castor seeds passing makes the form ready for the bed, not settled; one screen on one body never refutes a form. The frozen-address control at the dial's level means the function needs no learned address (a report, not a failure).

**Length:** about 30 minutes a Castor arm and 46 the lead-body arm on an RTX PRO 6000: about 3.8 hours for the seven on one card; split `NAMES` across two cards for about two. The cell can be stopped and rerun (each arm resumes from the repo, finished arms are read back).

**Token:** `HF_TOKEN` with write access to AbstractPhil/alephllm-nano-beatrix-training (the records land under `v1_lastfix/<arm>/`, beside the earlier screens').


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.34"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import lastfix_screen as LS
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 34), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/nano_head_dial'                      # the session folder: the starts, the runs, the records
REPO = LS.REPO                                        # AbstractPhil/alephllm-nano-beatrix-training, under v1_lastfix/<arm>/
NAMES = ['c-dial-s3', 'c-dial-s4', 'c-dialfrz-s3', 'c-dialraw-s3', 'c-dialplus-s3', 'c-pergain-s3', 'pdc-dial-s1']
REFS = ['c-head-adamw0.1', 'c-head-adamw0.1-s4', 'pdc-head-adamw0.1', 'c-bare-s3', 'c-bare-s4', 'pdc-bare']   # the reference and plain runs (read from the repo, never retrained here)
MAX_HOURS = None                                      # this session's budget in hours (None: until every arm is done)
print('arms:', ', '.join(NAMES), '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: every arm's start (checked against a fresh build), its optimizer split, the length, the push probe. Trains nothing.
import json, time
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.train.optim import build_optimizers
births = {n: LS.write_arm_birth(LS.BY_NAME[n], f'{OUT}/births') for n in NAMES}
step_s = {'nano-castor-v1': 0.299, 'nano-pollux-dual-canon-v1': 0.493}      # the v1 bench on an RTX PRO 6000 at 16 x 4 (2026-10-14)
for n in NAMES:
    a = LS.BY_NAME[n]
    t = LS.arm_preset(a, births[n]).train
    m = AlephLM(LS.arm_cfg(a))
    opts = build_optimizers(m, t.muon_lr, t.muon_momentum, t.adam_lr, decay_prefixes=t.decay_params, decay_wd=t.decay_wd)
    g = opts[1].param_groups[1]
    print(f"{n}: {a['craft']} seed {a['seed']} (reference {a.get('ref') or LS.PLAIN[(a['craft'], a['seed'])]}; head {a.get('head_mode') or 'dual'}{' over a frozen address' if a.get('addr_frozen') else ''}); {sum(p.numel() for p in g['params']):,} "
          f"readout weights on {type(opts[1]).__name__} at decay {g['weight_decay']}; {sum(p.numel() for p in opts[0].param_groups[0]['params']):,} on Muon")
mins = sum((4960 * step_s[LS.BY_NAME[n]['craft']] + 12 * 25) / 60 for n in NAMES)
print(f'-> about {mins / 60:.1f} h on an RTX PRO 6000 (training plus a full eval and the gauge every 500 steps)')
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'{LS.PREFIX}/preflight/push_probe_head_dial.json'
    api.upload_file(path_or_fileobj=json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                                'card': torch.cuda.get_device_name(0), 'arms': NAMES}).encode(),
                    path_in_repo=path, repo_id=REPO, commit_message='head dial screen: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE SCREEN: arm by arm; the trainer's own lines and the screen's gauge lines ([lastfix]) print as they come. Rerun the cell to
# continue (a new session too): each arm resumes from the repo, finished arms are read back. The stop button saves the running arm first.
RECS = LS.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, max_hours=MAX_HOURS)
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display

def with_refs(recs):
    out = {}
    for n in REFS:
        try:
            out[n] = json.load(open(hf_hub_download(REPO, f'{LS.PREFIX}/{n}/screen_record.json', token=HF_TOKEN or False), encoding='utf-8'))
        except Exception as e:  # noqa: BLE001
            print('no record yet for', n, type(e).__name__)
    out.update(recs)
    return out
display(Markdown('**At 1,500 (after the warmup)**\n\n' + LS.table(with_refs(RECS), 1500)))
display(Markdown('**At the end (each arm\'s last read)**\n\n' + LS.table(with_refs(RECS))))

In [ ]:
# C4 — THE CURVES AND THE TABLES from the repo (every session; tokenless: this session's files). Trains nothing.
import json
from IPython.display import Markdown, display
recs, logdir = {}, f'{OUT}/runs'
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                            allow_patterns=[f'{LS.PREFIX}/{n}/runs/*' for n in REFS + NAMES] + [f'{LS.PREFIX}/{n}/screen_record.json' for n in REFS + NAMES])
    logdir = f'{hub}/{LS.PREFIX}'
    for n in REFS + NAMES:
        p = f'{hub}/{LS.PREFIX}/{n}/screen_record.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
else:
    for n in NAMES:
        p = f'{OUT}/lastfix/{n}.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
display(Markdown('**At 1,500 (after the warmup)**\n\n' + LS.table(recs, 1500)))
display(Markdown('**Each arm\'s last read**\n\n' + LS.table(recs)))
%load_ext tensorboard
%tensorboard --logdir {logdir}

### Reading the result

Each arm is read against its reference run (the table's "vs its plain run" column: the readout-on-AdamW run of the same body and start).

| read | the reference run | an arm that works |
|---|---|---|
| the head's toggle (`lastwrite/head_toggle_bits`) | +0.0003 | at least +0.010 |
| bits a byte at 4,960 | the reference | within 0.0029, or better |
| share after the final norm | 0.28-0.30 | below the reference's |
| the dial's spread / saturation / corr(-H) | - | spread above 0, saturation under 0.5, corr above 0 |
| the frozen-address control (`c-dialfrz-s3`) | - | at the dial's level: the function needs no learned address (a report, not a failure) |
| the carrying read beside the dial (`c-dialplus-s3`: `lastwrite/head_w_s_fro`) | 6-9, asleep | whether it wakes once the dial exists |

### Changing the screen
| to | the edit in C1 |
|---|---|
| one card of two | `NAMES = NAMES[:4]` on the first card, `NAMES = NAMES[4:]` on the second |
| the primary pair only | `NAMES = ['c-dial-s3', 'c-dial-s4']` |
| a session budget | `MAX_HOURS = 1` |
